# Post Hoc Interpretation of JHU-Transferred HF Clusters in All of Us

This notebook characterizes the four All of Us (AoU) heart failure clusters assigned by the transferred Johns Hopkins University (JHU) VaDE model. It combines descriptive clinical visualizations with an XGBoost surrogate classifier and SHAP analysis to explore features associated with cluster assignments.

## Required inputs

- `feature_matrix_wide_deaths.csv`: Patient-level AoU feature matrix.
- `covariate_id_name_map.csv`: Mapping from covariate IDs to JHU-compatible feature names.
- `Outputs/vade_assignments/patient_cluster_assignments_jhu_transfer.parquet`: Previously generated VaDE cluster assignments.

## Outputs and interpretation

Figures are saved as SVG files in `../plots/`. Radar coordinates are also exported as `../plots/radar_coordinates.csv`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import re
import textwrap

import shap
from shap.plots._style import style_context
import matplotlib as mpl
import matplotlib.pyplot as plt
import xgboost
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

from pathlib import Path

shap.initjs()

seed = 42

In [ ]:
def fit_lab_scaler(train_df):
    """Per-lab robust location/scale (median / IQR) fit on TRAIN rows only.

    Args:
        train_df: training-row slice of the feature frame.

    Returns:
        dict with cols (the non-binary columns), med and iqr.
    """
    cols = [c for c in train_df.columns
            if not set(train_df[c].dropna()).issubset({0, 1, 0.0, 1.0})]
    med = train_df[cols].median()
    iqr = (train_df[cols].quantile(0.75) - train_df[cols].quantile(0.25)).replace(0, 1.0)
    return {"cols": cols, "med": med, "iqr": iqr}

In [ ]:
def encode_measurements(df, scaler, clip=4.0):
    """
    Robustly z-score continuous measurements.
    Keep binary features unchanged.
    Missing continuous values become 0, with mask=0 so they are excluded
    from reconstruction loss.
    """
    numeric_cols = scaler["cols"]
    median = scaler["med"]     
    iqr = scaler["iqr"]

    numeric = df[numeric_cols]
    observed_numeric = numeric.notna()

    numeric_scaled = (
        ((numeric - median) / iqr)
        .clip(-clip, clip)
        .where(observed_numeric, 0.0)
    )
    numeric_scaled.columns = [f"prof_{col}" for col in numeric_cols]

    binary = df.drop(columns=numeric_cols)

    feats = pd.concat([binary, numeric_scaled], axis=1)

    mask = pd.concat(
        [
            pd.DataFrame(1, index=df.index, columns=binary.columns),
            observed_numeric.astype(int).set_axis(numeric_scaled.columns, axis=1),
        ],
        axis=1,
    )[feats.columns]

    return feats, mask

## Load Features
Feature preprocessing is performed identically to before, see `verification.ipynb` for more information.

In [ ]:
DATASET = "heartfailure-aou"
FEATURES_PATH = 'feature_matrix_wide_deaths.csv'
COVMAP_PATH = 'covariate_id_name_map.csv'
ASSIGNMENTS_PATH = 'Outputs/vade_assignments/patient_cluster_assignments_jhu_transfer_deaths.parquet'

# --- load + rename columns to JHU-comparable covariate names ---
features_df = pd.read_csv(FEATURES_PATH).set_index('person_id')
features_df = features_df.loc[:, ~features_df.columns.duplicated()].copy()

cov_map_df = pd.read_csv(COVMAP_PATH)
covariate_map = cov_map_df.set_index('covariate_id')['remapped_name'].to_dict()
features_df.columns = [covariate_map.get(int(float(c)), str(c)) for c in features_df.columns]
features_df = features_df.loc[:, ~features_df.columns.duplicated()].copy()

# --- scale/encode exactly as in the VaDE training notebook ---
ids_train, _ = train_test_split(features_df.index, test_size=0.20, random_state=seed)
scaler = fit_lab_scaler(features_df.loc[ids_train])
feats_df, mask_df = encode_measurements(features_df, scaler, clip=4.0)

# drop rare binaries / all-zero features, same as the VaDE training notebook
binary_cols = [c for c in feats_df.columns if not c.startswith("prof_")]
keep_binary = feats_df[binary_cols].sum(axis=0) >= 10
feats_df = feats_df.loc[:, list(keep_binary[keep_binary].index) +
                        [c for c in feats_df.columns if c.startswith("prof_")]]
mask_df = mask_df.loc[:, feats_df.columns]
nonzero = (feats_df != 0).any(axis=0)
feats_df = feats_df.loc[:, nonzero]
mask_df = mask_df.loc[:, feats_df.columns]

# cluster assignments
assignments_df = pd.read_parquet(ASSIGNMENTS_PATH)
cluster = assignments_df.set_index('person_id')['cluster']

# join with feature dataframe by patient id
df = feats_df.merge(cluster, left_index=True, right_index=True)
df.head()


In [ ]:
X = df.drop(columns=['cluster'])
X.columns = X.columns.str.replace(r'[\[\]<>]', '', regex=True)  # XGBoost doesn't like [] in column names
y = df['cluster']

# create a train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
d_train = xgboost.DMatrix(X_train, label=y_train)
d_test = xgboost.DMatrix(X_test, label=y_test)


## Helper Functions
Every plotting, naming and feature-selection helper used below is defined in this section. Palette and
style constants, then label formatting, then one cell per figure family: measurement
ridgelines, radar plots, condition and drug classes, and the surrogate model and SHAP figures.

Typography is global: `FONT_FAMILY` names a [Google font](https://fonts.google.com/) that pyfonts
downloads and caches, `FONT_SIZE` sets the base size, and `set_style()` registers both with matplotlib
alongside the palette. Pass another `load_google_font(...)` to `set_style` to change typeface anywhere. Every figure sizes its text relative to that base, so `set_style(size=14)` or an
edit to `FONT_SIZE` rescales the whole notebook. `RIDGE_XLIM` fixes the ridgeline z-score window.

In [ ]:
K = 4
CLUSTER_COLORS = ["#568f8b", '#cd7e59', '#d15252', '#ddb247']
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#898781', '#e1e0d9', '#fcfcfb'
NOTE = '#454440'
SEQUENTIAL = LinearSegmentedColormap.from_list(
    'seq', ['#fcfcfb', '#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#104281'])

FONT_SIZE = 11
font = "DejaVu Sans" 
BANDS = ((0.95, 0.62), (0.80, 0.34), (0.50, 0.0))
LIFT_CLIP = 4.0
ODDS_CLIP = 16.0
RIDGE_XLIM = (-2, 3)

CLUSTER_NAMES = {
    1: "Hypertensive Pregnancy Analog",
    2: "Cardiorenal Analog",
    3: "Autoimmune Analog",
    4: "Cardiometabolic and Ischemic Analog",
}

def set_style(font=font, size=FONT_SIZE):
    plt.rcParams.update({
        'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
        'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK, 'text.color': INK,
        'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.grid': False,
        'axes.spines.top': False, 'axes.spines.right': False, 'font.size': size,
        'axes.titlesize': 'large', 'axes.labelsize': 'medium',
        'xtick.labelsize': 'small', 'ytick.labelsize': 'small',
        'legend.fontsize': 'small', 'figure.titlesize': 'x-large',
    })

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Helvetica', 'Arial', 'sans-serif']
set_style()

In [ ]:
LOINC_PROPERTIES = ('Mass/volume', 'Moles/volume', 'Units/volume', 'Number/volume',
                    'Enzymatic activity/volume', 'Volume Rate/Area', 'Mass Ratio', '#/volume')

UNIT_TOKENS = {
    'picogram': 'pg', 'nanogram': 'ng', 'microgram': 'µg', 'milligram': 'mg', 'gram': 'g',
    'milliliter': 'mL', 'deciliter': 'dL', 'microliter': 'µL', 'liter': 'L',
    'millimole': 'mmol', 'nanomole': 'nmol', 'milliequivalent': 'mEq',
    'international_unit': 'IU', 'micro-international_unit': 'µIU', 'microunit': 'µU',
    'unit': 'U', 'minute': 'min', 'percent': '%', 'thousand': '10³',
    'millimeter_mercury_column': 'mmHg', 'inch_(US)': 'in', 'ounce_(avoirdupois)': 'oz',
    'gram_of_creatinine': 'g creatinine', '1.73_square_meter': '1.73 m²',
}


def cluster_label(k):
    return f'{k + 1} · {CLUSTER_NAMES[k + 1]}'


def pretty_feature(name, width=44):
    label = re.sub(r'\s*\(.*\)$', '', pretty(name, 999))
    label = re.sub(r'\[[^\]]*\]', '', label)
    for prop in LOINC_PROPERTIES:
        label = label.replace(prop, '')
    label = re.split(r'\s+in\s+(?:Serum|Plasma|Blood)', label)[0]
    label = label.title()
    return label if len(label) <= width else label[:width - 1] + '…'


def wrap_label(name, width=26, per_line=14):
    label = re.sub(r'\s*\(.*\)$', '', re.sub(r'\[[^\]]*\]', '', pretty(name, 999)))
    label = re.split(r'\s+in\s+(?:Serum|Plasma|Blood)', ' '.join(label.split()))[0]
    if len(label) > width:
        label = label[:width - 1] + '…'
    return '\n'.join(textwrap.wrap(label, per_line, break_long_words=False))


def axis_label(feature, scales, base='Robust z-score'):
    unit = pretty_unit(feature) if has_scale(feature, scales) else ''
    return f'{base} ({unit})' if unit else base


def tick_labels(ticks, feature, scales):
    if not has_scale(feature, scales):
        return [f'{z:g}' for z in ticks]
    return [f'{z:g}\n{raw_value(z, feature, scales)}' for z in ticks]


# Make names more readable
def pretty(name, width=52):
    label = name[5:] if name.startswith('prof_') else name
    label = label.replace('_', ' ')
    return label if len(label) <= width else label[:width - 1] + '…'


def pretty_unit(name):
    match = re.search(r'\(([^()]*(?:\([^()]*\))?[^()]*)\)$', name)
    if not match:
        natural = re.search(r'_in_([A-Za-z]+)$', name)
        return natural.group(1) if natural else ''
    return '/'.join(UNIT_TOKENS.get(part, part.replace('_', ' '))
                    for part in match.group(1).split('_per_'))


def has_scale(feature, scales):
    return scales is not None and feature in scales.index


def raw_value(z, feature, scales):
    center, spread, log = scales.loc[feature, ['center', 'spread', 'log']]
    value = center + z * spread
    return compact_value(10 ** value if log else value)


def compact_value(value):
    if abs(value) >= 1e6:
        return f'{value / 1e6:,.3g}M'
    if abs(value) >= 1e4:
        return f'{value / 1e3:,.3g}k'
    return format_value(value)


def format_value(value):
    return f'{value:,.0f}' if abs(value) >= 100 else f'{value:,.3g}'


def plot_path(name, folder='../plots'):
    directory = Path(folder)
    directory.mkdir(parents=True, exist_ok=True)
    return directory / name


def slug(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')


def shade(color, amount):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)

In [ ]:
def plot_ridgeline(feature, feats, mask, y, overlap=0.18, window=1.0, scales=None):
    fig, axes = plt.subplots(K, 1, figsize=(9.5, 1.15 * K + 1.6))
    draw_ridges(axes, feature, feats, mask, y, labels=True, window=window, scales=scales)
    band_legend(fig, MUTED,  loc='upper right', bbox_to_anchor=(0.985, 1.0), ncol=4,
                compact=True)
    fig.suptitle(pretty(feature, 90), x=0.012, ha='left', fontsize='large', fontweight='bold')
    fig.subplots_adjust(left=0.27, right=0.98, top=0.85, hspace=-overlap)
    return fig


def plot_measurement_grid(features, feats, mask, y, ncols=4, overlap=0.21, window=1.0,
                          scales=None):
    nrows = int(np.ceil(len(features) / ncols))
    height = 4.1 * nrows + 1.0
    fig = plt.figure(figsize=(4.8 * ncols, height))
    outer = fig.add_gridspec(nrows, ncols, hspace=0.52, wspace=0.14,
                             left=0.03, right=0.985, top=0.855, bottom=0.62 / height)
    for i, feature in enumerate(features):
        inner = outer[i // ncols, i % ncols].subgridspec(K, 1, hspace=-overlap)
        axes = [fig.add_subplot(inner[k]) for k in range(K)]
        draw_ridges(axes, feature, feats, mask, y, window=window, scales=scales)
        axes[0].set_title(pretty_feature(feature), loc='center', fontsize='large', pad=8,
                          fontweight='bold')
    
    cluster_legend(fig)
    band_legend(fig, MUTED, loc='upper right', bbox_to_anchor=(0.985, 1.0), ncol=4,
                compact=True)
    return fig


def unclipped_measurements(features_path, covmap_path, reference, log=(), seed=seed):
    frame = log_transform(named_features_ours(features_path, covmap_path), log)
    ids_train, _ = train_test_split(frame.index, test_size=0.2, random_state=seed)
    scaler = fit_lab_scaler(frame.loc[ids_train])
    feats, obs = encode_measurements(frame, scaler, clip=np.inf)
    # NOTE: JHU merges duplicate unit-recorded versions of the same lab here via
    # merge_unit_splits(feats, obs, metadata); that function lives in JHU's `data`
    # module and depends on JHU's long-format OMOP metadata, which AoU's already-
    # deduplicated wide feature matrix + covariate map doesn't have an equivalent
    # for. There's nothing to merge here since named_features_ours doesn't split
    # units to begin with.
    columns = [c for c in reference.columns if c in feats.columns]
    return (feats.reindex(index=reference.index, columns=columns),
            measurement_scales(scaler, log).reindex(columns))


def draw_ridges(axes, feature, feats, mask, y, labels=False, window=1.0, cut=3.0, scales=None):
    obs = [observed_values(feats, mask, y, feature, k) for k in range(K)]
    pooled = np.concatenate([v for v in obs if v.size])
    lo, hi = span(pooled, window)
    top = 0.0
    for k, (ax, v) in enumerate(zip(axes, obs)):
        if v.size > 1:
            draw_ridge(ax, v, CLUSTER_COLORS[k], lo, hi, cut)
            top = max(top, ax.get_ylim()[1])
    for k, (ax, v) in enumerate(zip(axes, obs)):
        ax.axvline(0.0, color=MUTED, lw=0.9, ls=(0, (4, 4)), zorder=2)
        ax.axhline(0, color=INK, lw=1.1, clip_on=False, zorder=5)
        draw_intervals(ax, v, CLUSTER_COLORS[k], -0.16 * top)
        ax.set_xlim(*RIDGE_XLIM)
        ax.set_ylim(-0.30 * top, top * 1.05)
        ax.set_yticks([]); ax.set_ylabel('')
        ax.patch.set_alpha(0)
        ax.spines[:].set_visible(False)
        ax.tick_params(bottom=False, labelbottom=False)
        if labels:
            ax.text(-0.02, 0.30, f'{CLUSTER_NAMES[k + 1]}\nn = {v.size}', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize='large', fontweight='bold',
                    color='black')
    if labels:
        axes[0].text(0.0, 1.0, ' Cohort median', transform=axes[0].get_xaxis_transform(),
                     fontsize='x-small', color=NOTE, va='top')
    ticks = np.arange(np.ceil(RIDGE_XLIM[0]), np.floor(RIDGE_XLIM[1]) + 0.5)
    axes[-1].set_xticks(ticks)
    axes[-1].set_xticklabels(tick_labels(ticks, feature, scales))
    axes[-1].set_xlabel(axis_label(feature, scales), fontsize='medium', color=INK)
    axes[-1].tick_params(bottom=True, labelbottom=True, labelsize='small', colors=INK)


def draw_ridge(ax, v, color, lo, hi, cut=3.0):
    inside = windowed(v, lo, hi)
    if inside.size < 2:
        return
    below, above = int(np.sum(v < lo)), int(np.sum(v > hi))
    x, density = ridge_density(inside, cut, tails=(below == 0, above == 0))
    ax.fill_between(x, density, color=color, alpha=0.85, linewidth=0, zorder=3)
    ax.plot(np.r_[x[0], x, x[-1]], np.r_[0.0, density, 0.0], color=SURFACE, lw=1.6, zorder=4)


def draw_intervals(ax, v, color, y):
    if v.size < 2:
        return
    for level, fade in BANDS:
        q = np.quantile(v, [(1 - level) / 2, (1 + level) / 2])
        ax.plot(q, [y, y], color=shade(color, fade), lw=5.5, solid_capstyle='butt', zorder=7)
    ax.scatter([np.median(v)], [y], s=16, color=INK, edgecolor=SURFACE, linewidth=0.8, zorder=8)


def band_legend(target, color, loc='upper left', compact=False, **kwargs):
    handles = [plt.Line2D([], [], color=shade(color, fade), lw=5.5,
                          label=f'{level:.0%}' + ('' if compact else ' of patients'))
               for level, fade in BANDS]
    handles.append(plt.Line2D([], [], ls='', marker='o', markersize=5, color=INK, label='Cluster Median'))
    target.legend(handles=handles, loc=loc, frameon=True, facecolor=SURFACE, edgecolor=GRID,
                  fontsize='x-small', labelcolor=INK, handlelength=1.6, borderpad=0.7,
                  labelspacing=0.5, **kwargs)


def cluster_legend(fig):
    handles = [plt.Line2D([], [], marker='s', ls='', markersize=9, color=CLUSTER_COLORS[k],
                          label=f'{k+1} · {CLUSTER_NAMES[k + 1]}') for k in range(K)]
    fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.03, 1.0), ncol=K,
               frameon=False, fontsize='small', labelcolor=INK, handletextpad=0.4,
               columnspacing=1.6)


def log_transform(frame, patterns):
    frame = frame.copy()
    for column in frame.columns:
        if any(str(p).lower() in column.lower() for p in patterns):
            frame[column] = np.log10(frame[column].where(frame[column] > 0))
    return frame


def measurement_scales(scaler, patterns):
    logged = [any(str(p).lower() in c.lower() for p in patterns) for c in scaler['cols']]
    scales = pd.DataFrame({'center': scaler['med'], 'spread': scaler['iqr'],
                           'log': pd.Series(logged, index=scaler['cols'])})
    return scales.rename(index=lambda c: f'prof_{c}')


def named_features_ours(features_path, covmap_path):
    """AoU equivalent of JHU's named_features(): load the wide feature matrix and
    remap covariate_id columns to the same human-readable covariate names JHU uses,
    via the covariate_id_name_map.csv produced during feature extraction."""
    frame = pd.read_csv(features_path).set_index('person_id')
    frame = frame.loc[:, ~frame.columns.duplicated()].copy()

    cov_map_df = pd.read_csv(covmap_path)
    covariate_map = cov_map_df.set_index('covariate_id')['remapped_name'].to_dict()
    frame.columns = [covariate_map.get(int(float(c)), str(c)) for c in frame.columns]

    return frame.loc[:, ~frame.columns.duplicated()].copy()


def ridge_density(v, cut=3.0, tails=(True, True), points=512, bw_adjust=0.9):
    kde = gaussian_kde(v)
    kde.set_bandwidth(kde.factor * bw_adjust)
    bandwidth = float(np.sqrt(kde.covariance.squeeze()))
    x = np.linspace(v.min() - cut * bandwidth * tails[0],
                    v.max() + cut * bandwidth * tails[1], points)
    return x, kde(x)


def observed_values(feats, mask, y, feature, k):
    rows = (y.values == k) & (mask[feature].reindex(feats.index).values == 1)
    return feats.loc[rows, feature].values


def span(v, coverage=1.0, pad=0.06, reach=0.25):
    tail = (1 - coverage) / 2
    lo, hi = np.quantile(v, [tail, 1 - tail])
    width = hi - lo
    lo = v.min() if lo - v.min() <= reach * width else lo
    hi = v.max() if v.max() - hi <= reach * width else hi
    return lo - pad * width, hi + pad * width


def windowed(v, lo, hi):
    return v[(v >= lo) & (v <= hi)]

In [ ]:
def plot_cluster_radars(conditions, feats, y, scale='odds', ncols=2):
    conditions = resolve_conditions(conditions, feats)
    table = cluster_value_table(conditions, feats, y, scale)
    nrows = int(np.ceil(K / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.6 * ncols, 5.2 * nrows),
                             subplot_kw={'projection': 'polar'})
    for k, ax in enumerate(axes.flat):
        draw_radar(ax, table[k].values, conditions, color=CLUSTER_COLORS[k],
                   extent=table.values)
        ax.set_title(f'{k + 1} · {CLUSTER_NAMES[k + 1]}   n = {(y.values == k).sum()}',
                     fontsize='medium', fontweight='bold', color='black', pad=30)
    fig.suptitle(f'Recorded Conditions and Drugs by Cluster · {scale_title(scale)}',
                 x=0.5, y=0.985, fontsize='large', fontweight='bold')
    fig.subplots_adjust(top=0.90, bottom=0.03, hspace=0.16, wspace=0.46)
    return fig


def plot_combined_radar(conditions, feats, y, scale='lift'):
    conditions = resolve_conditions(conditions, feats)
    table = cluster_value_table(conditions, feats, y, scale)
    angles = ring_angles(len(conditions))
    fig, ax = plt.subplots(figsize=(8.4, 8.2), subplot_kw={'projection': 'polar'})
    draw_radar(ax, None, conditions, extent=table.values)
    for k in range(K):
        draw_polygon(ax, angles, table[k].values, CLUSTER_COLORS[k], alpha=0.07,
                     label=f'{k + 1} · {CLUSTER_NAMES[k + 1]}')
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, 0.01), frameon=False,
               fontsize='medium', labelcolor=INK, ncol=2, handletextpad=0.6,
               columnspacing=1.8)
    fig.subplots_adjust(top=0.90, bottom=0.16, left=0.10, right=0.90)
    return fig


def select_categorical_features(feats, y, n=10, floor_rate=5, ceiling_rate=95, by='points',
                                min_members=15):
    rates = cluster_prevalence([c for c in feats.columns if not c.startswith('prof_')],
                               feats, y)
    keep = rates['cohort'].between(floor_rate, ceiling_rate)
    keep &= feats[rates.index].sum() >= min_members
    rates = rates[keep]
    ranked = {k: feature_scores(rates, k, by).sort_values(ascending=False).index
              for k in range(K)}
    picked = []
    while len(picked) < n:
        for k in range(K):
            nxt = [f for f in ranked[k] if f not in [p for p, _ in picked]]
            if nxt and len(picked) < n:
                picked.append((nxt[0], k))
    return [f for f, _ in sorted(picked, key=lambda pair: pair[1])]


def draw_radar(ax, v, conditions, color=None, extent=None):
    angles = ring_angles(len(conditions))
    spread = np.ravel(extent if extent is not None else v)
    floor, ceiling, ticks = radial_limits(spread)
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    ax.set_ylim(floor, ceiling)
    ax.vlines(angles, floor, ceiling, color=GRID, ls=(0, (4, 4)), lw=0.8, zorder=1)
    if v is not None:
        draw_polygon(ax, angles, v, color)
    ax.plot(np.linspace(0, 2 * np.pi, 200), np.zeros(200), color=MUTED, lw=1.1, zorder=4)
    style_radar(ax, angles, conditions, ticks)


def draw_polygon(ax, angles, v, color, point_shape='o', alpha=0.16, lw=2.5, label=None):
    loop = np.append(angles, angles[0])
    values = np.append(v, v[0])
    ax.plot(loop, values, color=color, lw=lw, zorder=4, label=label, clip_on=False)
    ax.fill(loop, values, color=color, alpha=alpha, zorder=3)
    ax.scatter(angles, v, s=44, color=color, edgecolor=SURFACE, linewidth=0.8, zorder=5,
               marker=point_shape, clip_on=False)


def resolve_conditions(conditions, feats):
    columns = [c for c in feats.columns if not c.startswith('prof_')]
    lookup = {c.lower().replace('_', ' '): c for c in columns}
    resolved = []
    for name in conditions:
        key = str(name).lower().replace('_', ' ').strip()
        if key in lookup:
            resolved.append(lookup[key])
            continue
        hits = [c for k, c in lookup.items() if key in k]
        if len(hits) != 1:
            raise KeyError(f'{name!r} matches {len(hits)} categorical features: {hits[:6]}')
        resolved.append(hits[0])
    return resolved


def available_conditions(names, feats):
    columns = [c.lower().replace('_', ' ') for c in feats.columns]
    return [n for n in names
            if any(str(n).lower().replace('_', ' ').strip() in c for c in columns)]


def cluster_value_table(conditions, feats, y, scale):
    if scale == 'odds':
        x = feats[conditions]
        return pd.DataFrame({k: log_odds_ratio(x, y.values == k) for k in range(K)})
    if scale == 'lift':
        rates = cluster_prevalence(conditions, feats, y)
        return rates[list(range(K))].apply(log_lift, cohort=rates['cohort'])
    raise ValueError(f"scale must be 'odds' or 'lift', got {scale!r}")


def cluster_prevalence(conditions, feats, y):
    x = feats[conditions]
    columns = {k: x[y.values == k].mean() * 100 for k in range(K)}
    columns['cohort'] = x.mean() * 100
    return pd.DataFrame(columns)


def feature_scores(rates, k, by):
    if by == 'lift':
        return log_lift(rates[k], rates['cohort']).clip(lower=0)
    return (rates[k] - rates['cohort']).abs()


def log_lift(rate, cohort):
    ratio = (rate / cohort.replace(0, np.nan)).fillna(1.0)
    return np.log2(ratio.clip(1 / LIFT_CLIP, LIFT_CLIP))


def log_odds_ratio(x, member, correction=0.5):
    carriers, others = x[member].sum(), x[~member].sum()
    odds = ((carriers + correction) / (member.sum() - carriers + correction)
            / ((others + correction) / ((~member).sum() - others + correction)))
    return np.log2(odds.clip(1 / ODDS_CLIP, ODDS_CLIP))


def radial_limits(values, hole=0.30):
    limit = max(1.0, np.max(np.abs(values)) * 1.25)
    return -limit * (1 + hole), limit, ratio_rings(limit)


def ratio_rings(limit):
    return [t for t in np.arange(-4, 4.01, 1) if abs(t) <= limit]


def scale_title(scale):
    return 'Log2 Odds Ratio vs Rest' if scale == 'odds' else 'Log2 Enrichment vs Cohort Rate'


def style_radar(ax, angles, conditions, ticks):
    ax.set_yticks(ticks)
    ax.set_yticklabels([])
    ax.set_xticks(angles)
    ax.set_xticklabels([title_label(wrap_label(c)) for c in conditions], fontsize='medium')
    ax.tick_params(axis='x', pad=7, colors=INK)
    ax.grid(color=GRID, lw=0.8)
    ax.spines['polar'].set_visible(False)
    label_rings(ax, np.pi / len(angles), ticks)


def label_rings(ax, angle, ticks):
    for tick in ticks:
        ax.text(angle, tick, f'{tick:g}', ha='center', va='center', fontsize='x-small',
                color='black', fontweight='bold', zorder=6,
                bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.85, pad=1))


def ring_angles(n):
    return np.linspace(0, 2 * np.pi, n, endpoint=False)


def title_label(text):
    return re.sub(r'[A-Za-z][\w\-]*',
                  lambda m: m.group(0) if any(c.isupper() for c in m.group(0))
                  else m.group(0).capitalize(), text)

In [ ]:
DRUG_CLASSES = {
    'Loop diuretics': ['furosemide', 'bumetanide', 'torsemide', 'ethacrynic'],
    'Thiazide diuretics': ['hydrochlorothiazide', 'chlorothiazide', 'chlorthalidone',
                           'metolazone', 'indapamide'],
    'Aldosterone antagonists': ['spironolactone', 'eplerenone', 'finerenone'],
    'Beta blockers': ['metoprolol', 'carvedilol', 'atenolol', 'bisoprolol', 'propranolol',
                      'labetalol', 'nebivolol', 'sotalol', 'nadolol', 'betaxolol', 'esmolol',
                      'timolol'],
    'ACE inhibitors': ['lisinopril', 'enalapril', 'ramipril', 'captopril', 'benazepril',
                       'quinapril', 'fosinopril', 'perindopril'],
    'ARBs': ['losartan', 'valsartan', 'olmesartan', 'telmisartan', 'candesartan', 'irbesartan'],
    'ARNI': ['sacubitril'],
    'Statins': [r'statin\b'],
    'SGLT2 inhibitors': [r'gliflozin\b'],
    'GLP-1 agonists': [r'glutide\b'],
    'Insulins': [r'^insulin\b'],
    'Metformin': ['metformin'],
    'Anticoagulants': ['warfarin', 'apixaban', 'rivaroxaban', 'heparin', 'enoxaparin'],
    'Antiplatelets': ['aspirin', 'clopidogrel', 'ticagrelor', 'prasugrel'],
    'Antiarrhythmics': ['amiodarone', 'dofetilide', 'flecainide', 'digoxin'],
    'Nitrates and vasodilators': ['isosorbide', 'hydralazine', 'sildenafil'],
    'Estrogen or progestin': ['estradiol', 'estrogens', 'levonorgestrel', 'norethindrone',
                              'norgestimate', 'norgestrel', 'desogestrel', 'drospirenone',
                              'etonogestrel', 'norelgestromin', 'medroxyprogesterone',
                              'megestrol', 'progesterone'],
}

CONDITION_CLASSES = {
    'Lupus': [r'lupus(?![ _]anticoagulant)'],
    'Autoimmune disease': [
        r'lupus(?![ _]anticoagulant)', r'\bSLE\b', r'rheumatoid[ _]arthritis',
        r'systemic[ _]sclerosis', r'sj\w*gren', 'autoimmune', 'antiphospholipid',
    ],
    'Sarcoidosis': ['sarcoid'],
    'Sickle cell disease': [r'sickle[ _-]cell', r'hemoglobin[ _]ss'],
    'Anemia': ['anemia'],
    'Congenital heart disease': [
        r'congenital[ _]heart[ _]disease', r'ventricular[ _]septal[ _]defect',
        r'atrial[ _]septal[ _]defect', 'dextrocardia', r'patent[ _]foramen[ _]ovale',
    ],
    'Chronic kidney disease': [r'chronic[ _]kidney[ _]disease', r'end.stage[ _]renal'],
    'Type 2 diabetes': [r'type[ _]2[ _]diabetes'],
    'Type 1 diabetes': [r'type[ _]1[ _]diabetes'],
    'Ischemic heart disease': [r'myocardial[ _]infarction', 'angina', 'coronary'],
    'Prior MI': [r'^(?!percutaneous).*myocardial[ _]infarction'],
    'Heart failure': [r'heart[ _]failure', 'cardiomyopathy'],
    'Atrial fibrillation or flutter': [r'atrial[ _]fibrillation', r'atrial[ _]flutter'],
    'Systemic hypertension': [r'^(?!.*pulmonary).*hypertension', 'hypertensive'],
    'Pulmonary hypertension': [r'pulmonary[ _]hypertension'],
    'Pulmonary embolism': [r'pulmonary[ _]embolism'],
    'Interstitial lung disease': [r'interstitial[ _]lung[ _]disease'],
    'Endocarditis': ['endocarditis'],
    'Pregnancy or puerperium': ['pregnan', 'puerper', 'postpartum', 'eclampsia', 'gestation',
                                'childbirth', 'maternal', 'antepartum', 'peripartum',
                                'placenta'],
    'Hypertensive Pregnancy': ['eclampsia', 'maternal'],
    'Obesity': ['obesity', 'obese'],
    'Asthma or COPD': ['asthma', r'chronic[ _]obstructive'],
    'Thyroid disease': ['thyroid'],
}


def class_members(concepts, classes):
    return {name: [c for c in concepts if any(re.search(p, c, re.I) for p in patterns)]
            for name, patterns in classes.items()}


def concept_flags_ours(feats_df):
    binary_cols = [c for c in feats_df.columns if not c.startswith('prof_') and c != 'cluster']
    return feats_df[binary_cols].astype(int)

def class_flags_ours(feats_df, classes, index=None):
    flags = concept_flags_ours(feats_df)
    members = class_members(flags.columns, classes)
    grouped = pd.DataFrame({name: flags[cols].max(axis=1) for name, cols in members.items() if cols},
                            index=flags.index)
    return grouped.reindex(index).fillna(0).astype(int) if index is not None else grouped.astype(int)

def bmi_flag_ours(features_path, covmap_path, index=None, threshold=30.0):
    """AoU equivalent of JHU's bmi_flag(): pulls height/weight from the wide
    feature matrix + covariate map, unit-converts by matching whatever unit
    string appears in the (possibly underscore-joined) column name, and
    thresholds weight(kg) / height(m)**2 exactly as JHU does."""
    frame = named_features_ours(features_path, covmap_path)

    def find_measurement(concept, units):
        concept_norm = concept.lower().replace('_', ' ')
        candidates = [c for c in frame.columns
                      if concept_norm in c.lower().replace('_', ' ')]
        factors = {c: f for c in candidates
                   for unit, f in units.items()
                   if unit in c.lower().replace('_', ' ')}
        if not factors:
            return None
        converted = pd.concat([frame[c] * f for c, f in factors.items()], axis=1)
        return converted.bfill(axis=1).iloc[:, 0]

    height = find_measurement('Body height', {'inch': 0.0254, 'centimeter': 0.01, 'meter': 1.0})
    weight = find_measurement('Body weight', {'ounce': 0.0283495, 'pound': 0.453592,
                                              'gram': 0.001, 'kilogram': 1.0})
    if height is None or weight is None:
        print("No height/weight columns found in AoU features; BMI \u2265 30 flag will be all-zero.")
        flag = pd.Series(0, index=frame.index)
    else:
        flag = (weight / height ** 2 >= threshold).astype(int)
    return flag if index is None else flag.reindex(index).fillna(0).astype(int)

In [ ]:
def plot_fidelity(model, d_test, y_test):
    proba = model.predict(d_test, iteration_range=(0, model.best_iteration + 1))
    pred = proba.argmax(1)
    cm = confusion_matrix(y_test, pred, normalize='true')
    rng = np.random.default_rng(seed)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1, 1.15]})
    ax = axes[0]
    im = ax.imshow(cm, cmap=SEQUENTIAL, vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', fontsize='small',
                    color='#ffffff' if cm[i, j] > 0.55 else INK)
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xlabel('Predicted Cluster'); ax.set_ylabel('Assigned Cluster')
    ax.set_title('Surrogate Fidelity (Row-Normalised)', loc='left', fontsize='medium')
    ax.spines[:].set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.046, shrink=0.85).outline.set_visible(False)

    ax = axes[1]
    conf = proba.max(1)
    for k in range(K):
        m = y_test.values == k
        ax.scatter(k + rng.uniform(-.18, .18, m.sum()), conf[m], s=14,
                   color=CLUSTER_COLORS[k], alpha=.55, edgecolor=SURFACE, linewidth=.5)
    ax.axhline(0.5, color=MUTED, lw=1, ls=(0, (4, 3)))
    ax.set_xticks(range(K)); ax.set_xlabel('Assigned Cluster')
    ax.set_ylabel('Surrogate Confidence'); ax.set_ylim(0, 1)
    ax.set_title('Predicted Probability of the Assigned Cluster', loc='left', fontsize='medium')
    ax.yaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    fig.tight_layout()
    return pred


def plot_roc(model, d_test, y_test):
    proba = model.predict(d_test, iteration_range=(0, model.best_iteration + 1))
    curves, scores = ovr_roc(y_test.values, proba)

    fig, ax = plt.subplots(figsize=(6.4, 5.8))
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
    for k in range(K):
        fpr, tpr = curves[k]
        ax.plot(fpr, tpr, color=CLUSTER_COLORS[k], lw=1.9, zorder=3,
                label=f'{cluster_label(k)} (AUC = {scores[k]:.3f})')
    fpr, tpr = curves['micro']
    ax.plot(fpr, tpr, color=INK, lw=1.5, ls=(0, (5, 2)), zorder=2,
            label=f"Micro-Average (AUC = {scores['micro']:.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.005)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f"Surrogate ROC, One-vs-Rest \u00b7 Macro AUC = {scores['macro']:.3f}",
                 loc='left', fontsize='large', fontweight='bold')
    ax.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc='lower right', fontsize='small', labelcolor=INK)
    fig.tight_layout()

    index = [cluster_label(k) for k in range(K)] + ['Micro-Average', 'Macro-Average']
    values = [scores[k] for k in range(K)] + [scores['micro'], scores['macro']]
    return pd.Series(values, index=index, name='ROC-AUC')


def plot_global_importance(mean_abs, ranked, top=20):
    feats = ranked.head(top).index[::-1]
    vals = mean_abs.loc[feats]
    fig, ax = plt.subplots(figsize=(10, 0.34 * top + 1.2))
    left = np.zeros(len(feats))
    for k in range(K):
        w = vals[f'cluster {k}'].values
        ax.barh(range(len(feats)), w, left=left, height=.72, color=CLUSTER_COLORS[k],
                label=cluster_label(k), edgecolor=SURFACE, linewidth=1.4)
        left += w
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels([pretty_feature(f, 52) for f in feats], fontsize='small', color=INK)
    ax.set_xlabel('Mean |SHAP|')
    ax.set_title('Feature Importance by Cluster', loc='left', fontsize='large',
                 fontweight='bold')
    ax.xaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    ax.legend(frameon=False, ncol=2, loc='lower right', fontsize='small', labelcolor=INK)
    fig.tight_layout()


def plot_beeswarm_grid(sv, top=10):
    fig = plt.figure(figsize=(13, 9))
    for k in range(K):
        ax = fig.add_subplot(2, 2, k + 1)
        plt.sca(ax)
        shap.plots.beeswarm(sv[:, :, k], max_display=top, show=False,
                            color_bar=(k == 1), plot_size=None)
        ax.set_title(cluster_label(k), loc='left', fontsize='medium',
                     color='black', fontweight='bold')
        ax.tick_params(labelsize='x-small')
        ax.set_xlabel('SHAP (Log-Odds Toward This Cluster)', fontsize='small')
    fig.tight_layout()


def plot_dependence(sv, X, y, feature, highlight=True, ncols=K, panel=(3.4, 3.4)):
    j = list(X.columns).index(feature)
    nrows = int(np.ceil(K / ncols))
    fig, axes = plt.subplots(nrows, ncols, sharey=True, squeeze=False,
                             figsize=(panel[0] * ncols, panel[1] * nrows))
    for k, ax in enumerate(axes.flat):
        if k >= K:
            ax.set_visible(False)
            continue
        draw_dependence(ax, X[feature].values, sv.values[:, j, k], y, k, highlight)
        ax.set_title(cluster_label(k), loc='left', fontsize='small',
                     color='black', fontweight='bold')
        ax.set_xlabel('Feature Value (Robust z-score)', fontsize='small', color=INK)
        ax.tick_params(labelsize='small', colors=INK)
        ax.yaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    for row in axes:
        row[0].set_ylabel('SHAP (Log-Odds)', color=INK)
    if not highlight:
        axes[0, 0].legend(frameon=False, fontsize='x-small', labelcolor=INK, loc='upper left')
    fig.suptitle(f'{pretty_feature(feature, 70)} Per Cluster Distribution', x=.008, ha='left',
                 fontsize='large')
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.14 / nrows))
    return fig


def plot_exemplars(sv, X, y, proba, top=10):
    safe_top = min(top, X.shape[1] - 1)  # guard against max_display >= feature count
    for k in range(K):
        members = np.where(y.values == k)[0]
        i = members[proba[members, k].argmax()]
        with style_context(primary_color_positive=CLUSTER_COLORS[k],
                   primary_color_negative=MUTED,
                   secondary_color_positive=shade(CLUSTER_COLORS[k], .4),
                   secondary_color_negative=shade(MUTED, .4),
                   text_color=INK, hlines_color=GRID, vlines_color=MUTED,
                   tick_labels_color=MUTED):
            plt.figure()
            shap.plots.waterfall(sv[i, :, k], max_display=safe_top, show=False)
        fig = plt.gcf()
        fig.set_size_inches(9, 4.2)
        fig.suptitle(f'{cluster_label(k)} — Patient (p={proba[i, k]:.2f})',
                     x=.01, ha='left', fontsize='medium', color='black', fontweight='bold')
        plt.savefig(plot_path(f'{slug(cluster_label(k))}_waterfall.svg'), bbox_inches='tight')
        plt.show()


def draw_dependence(ax, values, shap_values, y, k, highlight):
    if highlight:
        others = y.values != k
        ax.scatter(values[others], shap_values[others], s=11, color=shade(MUTED, .62),
                   alpha=.85, edgecolor=SURFACE, linewidth=.3, zorder=2)
        members = y.values == k
        ax.scatter(values[members], shap_values[members], s=15, color=CLUSTER_COLORS[k],
                   alpha=.75, edgecolor=SURFACE, linewidth=.4, zorder=3)
    else:
        for c in range(K):
            m = y.values == c
            ax.scatter(values[m], shap_values[m], s=13, color=CLUSTER_COLORS[c], alpha=.6,
                       edgecolor=SURFACE, linewidth=.4,
                       label=f'{c} · {CLUSTER_NAMES[c + 1]}' if k == 0 else None)
    ax.axhline(0, color=MUTED, lw=.9, zorder=1)


def ovr_roc(y_true, proba):
    onehot = label_binarize(y_true, classes=range(K))
    curves, scores = {}, {}
    for k in range(K):
        fpr, tpr, _ = roc_curve(onehot[:, k], proba[:, k])
        curves[k], scores[k] = (fpr, tpr), auc(fpr, tpr)
    fpr, tpr, _ = roc_curve(onehot.ravel(), proba.ravel())
    curves['micro'], scores['micro'] = (fpr, tpr), auc(fpr, tpr)
    scores['macro'] = roc_auc_score(y_true, proba, multi_class='ovr', average='macro')
    return curves, scores


## Unclipped Measurements
Measurement ridgelines are drawn from robust z-scores refit without the training clip, so the tails
of each distribution stay visible.

In [ ]:
MEASURES, SCALES = unclipped_measurements(FEATURES_PATH, COVMAP_PATH, df,
                                          log=['Natriuretic_peptide'])


### Measurement Panel

In [ ]:
FEATURES = ['prof_Natriuretic_peptide.B_prohormone_N-Terminal_[Mass/volume]_in_Serum_or_Plasma_(picogram_per_milliliter)',
            'prof_age_in_years',
            'prof_Glomerular_filtration_rate_[Volume_Rate/Area]_in_Serum,_Plasma_or_Blood_by_Creatinine-based_formula_(MDRD)/1.73_sq_M_among_non_black_population_(milliliter_per_minute_per_1.73_square_meter)',
            'prof_Systolic_blood_pressure_(millimeter_mercury_column)']

# defensive check: not every JHU covariate survived AoU extraction (94% did, see
# extraction notes) -- fall back to whatever subset is actually present rather than
# hard-erroring on a KeyError.
missing = [f for f in FEATURES if f not in MEASURES.columns]
if missing:
    print(f"{len(missing)} of JHU's 4 panel features aren't in AoU's MEASURES; dropping:")
    for f in missing:
        print(f"  - {f}")
    FEATURES = [f for f in FEATURES if f not in missing]

plot_measurement_grid(FEATURES, MEASURES, mask_df, df['cluster'], scales=SCALES)
plt.savefig('../plots/ridgeplots_measurements.svg', bbox_inches='tight')
plt.show()


## Radar Plots

### Indibidual Conditions
Radar plots of 10 selected features map cluster identities onto recognizable condition prevalence. These are chosen at the individual condition level. Two scales are available: `scale='odds'` shows the log2 odds ratio of each cluster against the rest of the cohort, while `scale='lift'` shows the log2 enrichment of the cluster rate against the cohort rate. Both draw rings one log2 unit apart, labelled with the plotted value, and put the baseline of no difference on the dark circle.

Pass any list of conditions or drugs to these functions, for example `plot_cluster_radars(['type 2 diabetes mellitus', 'atorvastatin'], df, df['cluster'])`, names are matched case-insensitively against the categorical features, so the underscores in the column names are optional. 

In [ ]:
CONDITIONS = select_categorical_features(df, df['cluster'], n=10)

# plot_cluster_radars(CONDITIONS, df, df['cluster'])
# plt.show()

### Radar Plot by Class
Summarize groups of drugs and conditions into condition classes, then plot aggregated versions of the radar plots. These are used to get a higher level overview of the distribution of conditions and drugs across clusters.

In [ ]:
CLASSES = DRUG_CLASSES | CONDITION_CLASSES
CLASS_FLAGS = class_flags_ours(df, CLASSES, index=df.index)
CLASS_FLAGS['BMI ≥ 30'] = bmi_flag_ours(FEATURES_PATH, COVMAP_PATH, index=df.index)

In [ ]:
# Class panel: JHU's exact category set, in the requested order.
CLASS_PANEL = [
    'Hypertensive Pregnancy',
    'Prior MI',
    'BMI ≥ 30',
    'Autoimmune disease',
    'Sickle cell disease',
    'Chronic kidney disease',
    'Type 2 diabetes',
    'Antiplatelets',
    'Statins',
    'Pulmonary hypertension',
    'Beta blockers',
]

# sanity check before plotting: confirm every category actually has
# a nonzero column in CLASS_FLAGS
missing = [c for c in CLASS_PANEL if c not in CLASS_FLAGS.columns]
if missing:
    raise ValueError(f"These categories aren't in CLASS_FLAGS, check CLASSES dict: {missing}")

# JHU's cell mirrors this structure: individual per-cluster radars commented out,
# one combined radar (log2 lift vs cohort) rendered and saved.
# plot_cluster_radars(CLASS_PANEL, CLASS_FLAGS, df['cluster']).subplots_adjust(
    # top=0.86, bottom=0.04, hspace=0.34, wspace=0.46)
plot_combined_radar(CLASS_PANEL, CLASS_FLAGS, df['cluster'], scale='lift')
plt.savefig('../plots/radar_enrichment_combined.svg')
# plot_combined_radar(CLASS_PANEL, CLASS_FLAGS, df['cluster'], scale='odds')
plt.show()


In [ ]:
height_cols = [c for c in feats_df.columns if 'height' in c.lower()]
weight_cols = [c for c in feats_df.columns if 'weight' in c.lower()]
bmi_cols = [c for c in feats_df.columns if 'bmi' in c.lower() or 'body mass' in c.lower()]

print("Height-related columns in feats_df:")
for c in height_cols:
    print(" ", c)
print("\nWeight-related columns in feats_df:")
for c in weight_cols:
    print(" ", c)
print("\nBMI-related columns in feats_df:")
for c in bmi_cols:
    print(" ", c)

print(f"\ntotal columns in feats_df: {feats_df.shape[1]}")

In [ ]:
conditions = resolve_conditions(CLASS_PANEL, CLASS_FLAGS)

radar_values = cluster_value_table(
    conditions,
    CLASS_FLAGS,
    df["cluster"],
    scale="lift"
)

radar_coordinates = radar_values.copy()
radar_coordinates.insert(
    0,
    "angle_radians",
    ring_angles(len(conditions))
)

# Rows = categories; columns = angle plus cluster values
radar_coordinates.to_csv(
    "../plots/radar_coordinates.csv",
    index_label="category"
)

radar_coordinates

In [ ]:
saved = pd.read_csv(
    "../plots/radar_coordinates.csv",
    index_col="category"
)

conditions = saved.index.tolist()
angles = saved.pop("angle_radians").to_numpy()
radar_values = saved

fig, ax = plt.subplots(
    figsize=(8.4, 8.2),
    subplot_kw={"projection": "polar"}
)

draw_radar(
    ax,
    None,
    conditions,
    extent=radar_values.to_numpy()
)

for column in radar_values.columns:
    k = int(column)
    draw_polygon(
        ax,
        angles,
        radar_values[column].to_numpy(),
        CLUSTER_COLORS[k],
        alpha=0.07,
        label=f"{k + 1} · {CLUSTER_NAMES[k + 1]}"
    )

ax.legend()
plt.show()

## Train xGBoost Classifier

In [ ]:
params = {
    'objective': 'multi:softprob',
    'num_class': 4,
    'eval_metric': ['mlogloss', 'merror'],
    'eta': 0.05,
    'max_depth': 3,
    'min_child_weight': 1,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'lambda': 1,
    'device': 'cpu',       # AoU-environment addition, harmless elsewhere
    'tree_method': 'hist', # AoU-environment addition, harmless elsewhere
    'seed': seed,
}


model = xgboost.train(
    params,
    d_train,
    5000,
    evals=[(d_test, "test")],
    verbose_eval=5,
    early_stopping_rounds=20,
)

### Surrogate Fidelity

SHAP explains the supervised learning or 'surrogate' model, not the VaDE model. Therefore, we check the ability of the surrogate to recover the original clustering solution on a held out test set. Generally, xgboost is more conservative than the VaDE model at assigning patients to clusters, so we can trust the SHAP value explainability of the original VaDE model. This is still a posthoc explaination.

In [ ]:
pred = plot_fidelity(model, d_test, y_test)
print(classification_report(y_test, pred, digits=3))

### Discrimination (ROC-AUC)

In [ ]:
AUCS = plot_roc(model, d_test, y_test)
plt.savefig('../plots/roc_xgboost.svg', bbox_inches='tight')
plt.show()
AUCS

## SHAP Values

`explainer(X)` returns an `Explanation` of shape `(patients, features, clusters)`. Attributions
are in log-odds toward each cluster.

In [ ]:
explainer = shap.TreeExplainer(model)
sv = explainer(X)
sv.feature_names = [pretty_feature(f, 40) for f in X.columns]

cols = [f'cluster {k}' for k in range(K)]
mean_abs = pd.DataFrame(np.abs(sv.values).mean(0), index=X.columns, columns=cols)
ranked = mean_abs.sum(axis=1).sort_values(ascending=False)

print(f'{(ranked > 0).sum()} of {len(ranked)} features carry any attribution')
ranked.head(10)

## Global Importance

Total attribution per feature, split by the cluster it acts on. A tall single-colour bar is a
feature that separates one cluster; a mixed bar is one the model reuses everywhere.

In [ ]:
plot_global_importance(mean_abs, ranked)
plt.savefig('../plots/feature_importance_xgboost.svg')

## Per-Cluster Distributions (Beeswarm Plot)

Each point is one patient. Horizontal spread is how hard the feature pushes; colour is the
feature's own value, so a clean blue/red split means the effect is monotone in that feature.

In [ ]:
plot_beeswarm_grid(sv)
plt.savefig('../plots/beeswarm_xgboost.svg')

## Dependence

For a single feature, how its value maps to its effect on every cluster at once.

In [ ]:
plot_dependence(sv, X, y, ranked.index[0])
plt.savefig('../plots/creatinine_cluster_dependence.svg')

In [ ]:
# JHU hardcodes:
#   'prof_Platelets_#/volume_in_Blood_by_Automated_count_(thousand_per_microliter)'
# That exact remapped string isn't guaranteed to survive AoU's covariate-id remap
# 1:1, so find the platelet column by substring instead of hardcoding it.
platelet_candidates = [c for c in X.columns if 'platelet' in c.lower()]
if platelet_candidates:
    plot_dependence(sv, X, y, platelet_candidates[0])
    plt.savefig('../plots/platelets_cluster_dependence.svg')
else:
    print("No platelet measurement column found in X; skipping this dependence plot.")


## Individual Patient Waterfall

The most confidently assigned member of each cluster, decomposed from the cohort base rate to
that patient's log-odds. Generates a waterfall graph with per feature contributions.

In [ ]:
proba = model.predict(xgboost.DMatrix(X), iteration_range=(0, model.best_iteration + 1))
plot_exemplars(sv, X, y, proba)